# 5. Whole-Image vs. Quadrant Processing Time

## Objective
Compare the runtime of the same divide-by-64, multiply-by-64 quantization on the full image and on four quadrants.

The quadrant timing includes splitting the image, processing each region, and reassembling the result. The notebook first checks that both approaches produce identical pixels. Timing varies with hardware and system load.

## Run

Install the packages listed in the repository-level requirements.txt. Set IMAGE_PATH to a local image path to replace the sample input, then run the code cell.

In [ ]:
from pathlib import Path
from time import perf_counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
from skimage import data

# Set this to Path("path/to/image.jpg") to use your own image.
IMAGE_PATH = None
STEP = 64

if IMAGE_PATH is None:
    image = data.camera()
else:
    image = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(f"Could not read image: {IMAGE_PATH}")

def quantize(image: np.ndarray) -> np.ndarray:
    return ((image.astype(np.uint16) // STEP) * STEP).astype(np.uint8)

def quantize_by_quadrant(image: np.ndarray) -> np.ndarray:
    middle_y, middle_x = image.shape[0] // 2, image.shape[1] // 2
    top = np.concatenate(
        (quantize(image[:middle_y, :middle_x]), quantize(image[:middle_y, middle_x:])),
        axis=1,
    )
    bottom = np.concatenate(
        (quantize(image[middle_y:, :middle_x]), quantize(image[middle_y:, middle_x:])),
        axis=1,
    )
    return np.concatenate((top, bottom), axis=0)

def median_runtime_ms(function, repeats: int = 7, iterations: int = 25) -> float:
    """Return the median per-call time across batches after a warm-up call."""
    function()
    samples = []
    for _ in range(repeats):
        start = perf_counter()
        for _ in range(iterations):
            function()
        samples.append((perf_counter() - start) * 1000 / iterations)
    return float(np.median(samples))

whole_result = quantize(image)
quadrant_result = quantize_by_quadrant(image)
assert np.array_equal(whole_result, quadrant_result), "The two results must be identical."

whole_ms = median_runtime_ms(lambda: quantize(image))
quadrant_ms = median_runtime_ms(lambda: quantize_by_quadrant(image))
ratio = quadrant_ms / whole_ms if whole_ms else float("inf")

print(f"Image shape: {image.shape}")
print(f"Whole-image median: {whole_ms:.4f} ms per call")
print(f"Quadrant pipeline median: {quadrant_ms:.4f} ms per call")
print(f"Quadrant / whole-image time: {ratio:.2f}x")
print("Outputs identical: yes")
print("Quadrant timing includes splitting and reassembly.")

figure, axes = plt.subplots(1, 3, figsize=(18, 6))
panels = (
    (image, "Original"),
    (whole_result, f"Whole image\n{whole_ms:.3f} ms"),
    (quadrant_result, f"Four quadrants\n{quadrant_ms:.3f} ms"),
)
for axis, (result, title) in zip(axes, panels):
    axis.imshow(result, cmap="gray", vmin=0, vmax=255)
    axis.set_title(title)
    axis.axis("off")

figure.tight_layout()
plt.show()